In [1]:
s = "I love learning machine learning"

In [9]:
vocab = list(set(s.split())) #create a set of unique words in the string and convert it to a list
print(vocab)

['learning', 'love', 'I', 'machine']


In [10]:
import numpy as np

logitvec = np.array([0.2, 0.5, 0.8, 1.2]) #logit are the raw predictions of token probabilities

In [11]:
import torch
import torch.nn.functional as F
logitvec = F.softmax(torch.tensor(logitvec), dim=0).numpy() #softmax converts logits to probabilities

print(logitvec)

[0.14513242 0.19590827 0.2644485  0.39451081]


In [12]:
next_token_id = torch.argmax(torch.tensor(logitvec)).item() #get the index of the token with the highest probability
print(next_token_id)

3


In [13]:
next_token = vocab[next_token_id] #get the token corresponding to the index
print(next_token)

machine


In [14]:
s2 = "ArithmeticError is the base class for those built-in exceptions that are raised for various arithmetic errors, including OverflowError, ZeroDivisionError, and FloatingPointError.    "
vocab2 = list(set(s2.split())) #create a set of unique words in the string and convert it to a list
print(vocab2)

['the', 'arithmetic', 'exceptions', 'is', 'are', 'raised', 'those', 'for', 'base', 'built-in', 'OverflowError,', 'ArithmeticError', 'that', 'class', 'ZeroDivisionError,', 'various', 'including', 'errors,', 'and', 'FloatingPointError.']


In [15]:
for i in range(20):
    logitvec2 = torch.rand(len(vocab2)) #generate random logits for the second string

In [16]:
print(logitvec2)

tensor([0.9591, 0.3970, 0.6729, 0.1899, 0.6560, 0.4481, 0.9784, 0.4930, 0.5310,
        0.2174, 0.3553, 0.9981, 0.5475, 0.7069, 0.0424, 0.1814, 0.3619, 0.5563,
        0.7465, 0.2126])


In [17]:
logitvec2 = F.softmax(logitvec2, dim=0) #softmax converts logits to probabilities
print(logitvec2)

tensor([0.0753, 0.0429, 0.0565, 0.0349, 0.0556, 0.0452, 0.0767, 0.0472, 0.0491,
        0.0359, 0.0412, 0.0783, 0.0499, 0.0585, 0.0301, 0.0346, 0.0414, 0.0503,
        0.0609, 0.0357])


In [18]:
#we are trying to implement top k
#say k=10 here
#so now we will keep top 10 probabilities and set the rest to 0
k = 10
for i in range(len(logitvec2)):
    if logitvec2[i] < torch.topk(logitvec2, k).values[-1]: #if the probability is less than the kth highest probability
        logitvec2[i] = 0 #set it to 0
print(logitvec2)

tensor([0.0753, 0.0000, 0.0565, 0.0000, 0.0556, 0.0000, 0.0767, 0.0000, 0.0491,
        0.0000, 0.0000, 0.0783, 0.0499, 0.0585, 0.0000, 0.0000, 0.0000, 0.0503,
        0.0609, 0.0000])


In [19]:
next_token2_id = torch.multinomial(logitvec2, 1).item() #sample a token from the top k probabilities
print(next_token2_id)
next_token2 = vocab2[next_token2_id] #get the token corresponding to the index
print(next_token2)

13
class


In [21]:
for i, prob in enumerate(logitvec2):
    if prob > 0:
        print(f"Token: {vocab2[i]}, Probability: {prob.item()}")

Token: the, Probability: 0.07527415454387665
Token: exceptions, Probability: 0.056539442390203476
Token: are, Probability: 0.055594414472579956
Token: those, Probability: 0.07673931121826172
Token: base, Probability: 0.049062032252550125
Token: ArithmeticError, Probability: 0.07826578617095947
Token: that, Probability: 0.0498783141374588
Token: class, Probability: 0.05849780514836311
Token: errors,, Probability: 0.050318051129579544
Token: and, Probability: 0.06085614487528801


In [22]:
#Now lets implement top p sampling

#we need to sort the probabilities in descending order and then keep adding them until we reach the threshold p. Once we reach the threshold, we will set the rest of the probabilities to 0.

for i in range(20):
    logitvec2 = torch.rand(len(vocab2)) #generate random logits for the second string
    
logitvec2 = F.softmax(logitvec2, dim=0) #convert logits to probabilities
print(logitvec2)

tensor([0.0657, 0.0593, 0.0406, 0.0398, 0.0513, 0.0447, 0.0364, 0.0520, 0.0439,
        0.0370, 0.0535, 0.0385, 0.0462, 0.0347, 0.0673, 0.0775, 0.0470, 0.0526,
        0.0368, 0.0752])


In [23]:
#sort and then cumulative sum
sorted_probs, sorted_indices = torch.sort(logitvec2, descending=True)
cumulative_probs = torch.cumsum(sorted_probs, dim=0)
print(cumulative_probs)


tensor([0.0775, 0.1527, 0.2200, 0.2857, 0.3450, 0.3985, 0.4511, 0.5030, 0.5543,
        0.6013, 0.6476, 0.6923, 0.7362, 0.7768, 0.8166, 0.8552, 0.8921, 0.9289,
        0.9653, 1.0000])


In [25]:
#let p be 0.8 which means we will keep adding probabilities until we reach 0.8 and then set the rest to 0
p = 0.8
threshold_indices = torch.where(cumulative_probs <= p)[0]
print(threshold_indices)

#set the probabilities of the tokens that are not in the threshold_indices to 0
for i in range(len(logitvec2)):
    if i not in threshold_indices:
        logitvec2[i] = 0

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13])


In [26]:
next_word = np.random.choice(vocab, p=logitvec) #sample a word from the vocabulary based on the probabilities
print(next_word)

I
